# CONFIGURATION & IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import warnings

warnings.filterwarnings('ignore')

# Configuration
INPUT_DIR = '/kaggle/input/march-machine-learning-mania-2026/' # Adjust if necessary
RANDOM_STATE = 42
N_SPLITS = 5

# Check available files
print("Available Files:")
for file in os.listdir(INPUT_DIR):
    print(f" - {file}")

# LOAD AND EXPLORE DATASETS

In [ ]:
print("\n## Step 1: Load and Explore Datasets")

# Load core datasets (Assuming Men's data 'M' prefix, adjust to 'W' if needed)
try:
    teams = pd.read_csv(os.path.join(INPUT_DIR, 'MTeams.csv'))
    seasons = pd.read_csv(os.path.join(INPUT_DIR, 'MSeasons.csv'))
    regular_season = pd.read_csv(os.path.join(INPUT_DIR, 'MRegularSeasonCompactResults.csv'))
    tourney_results = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneyCompactResults.csv'))
    tourney_seeds = pd.read_csv(os.path.join(INPUT_DIR, 'MNCAATourneySeeds.csv'))
    sample_sub = pd.read_csv(os.path.join(INPUT_DIR, 'MSampleSubmissionStage1.csv'))
except FileNotFoundError as e:
    print(f"File not found: {e}. Please check INPUT_DIR and file names.")
    # For the sake of this script running without error in a demo, we would stop here.
    # In a real notebook, ensure paths are correct.
    raise e

# Basic Inspection
def inspect_df(df, name):
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape}")
    print(df.head(2))
    print(f"Missing Values:\n{df.isnull().sum()}")

inspect_df(teams, "Teams")
inspect_df(regular_season, "Regular Season Results")
inspect_df(tourney_results, "Tourney Results")
inspect_df(sample_sub, "Sample Submission")

# BUILD TEAM-LEVEL STATISTICS TABLE

In [ ]:
print("\n## Step 2: Build Team-Level Statistics")

def build_team_stats(results_df, seasons_list):
    """
    Aggregates stats per Team per Season.
    Returns: DataFrame with Season, TeamID, Wins, Losses, PointsScored, PointsAllowed, etc.
    """
    stats = []

    # We need to consider both WTeam and LTeam to get full stats for every team
    for season in seasons_list:
        season_data = results_df[results_df['Season'] == season]

        # Wins
        w_stats = season_data.groupby('WTeamID').agg(
            Wins=('WTeamID', 'count'),
            PointsScored=('WScore', 'sum'),
            PointsAllowed=('LScore', 'sum'),
            GamesPlayed=('WTeamID', 'count')
        ).reset_index().rename(columns={'WTeamID': 'TeamID'})

        # Losses
        l_stats = season_data.groupby('LTeamID').agg(
            Losses=('LTeamID', 'count'),
            PointsScored=('LScore', 'sum'),
            PointsAllowed=('WScore', 'sum'),
            GamesPlayed=('LTeamID', 'count')
        ).reset_index().rename(columns={'LTeamID': 'TeamID'})

        # Merge
        season_stats = pd.merge(w_stats, l_stats, on=['Season', 'TeamID'], how='outer').fillna(0)
        season_stats['Season'] = season
        stats.append(season_stats)

    full_stats = pd.concat(stats, ignore_index=True)
    full_stats['WinRate'] = full_stats['Wins'] / (full_stats['Wins'] + full_stats['Losses'] + 1e-6)
    full_stats['AvgScoreMargin'] = (full_stats['PointsScored'] - full_stats['PointsAllowed']) / (full_stats['GamesPlayed'] + 1e-6)

    return full_stats

# Get list of seasons available in training data
available_seasons = regular_season['Season'].unique()
team_stats = build_team_stats(regular_season, available_seasons)

print(f"Team Stats Shape: {team_stats.shape}")
print(team_stats.head())

# IMPLEMENT ELO RATING SYSTEM

In [ ]:
print("\n## Step 3: Implement Elo Rating System")

def calculate_elo_ratings(results_df):
    """
    Calculates Elo ratings for every team after every game.
    Returns a dictionary mapping (Season, TeamID, DayNum) -> Elo Rating
    For simplicity in feature engineering, we will map (Season, TeamID) -> End of Season Elo
    OR we calculate dynamically. Here we calculate dynamic Elo per game for accuracy.
    """
    # Sort by time
    df = results_df.sort_values(['Season', 'DayNum']).copy()

    # Initialize Elo dict: Key = TeamID, Value = Rating
    # We reset Elo every season usually, or carry over. Let's reset per season for simplicity.
    elo_ratings = {}
    history = [] # To store elo at time of game

    K_FACTOR = 32
    HOME_ADVANTAGE = 0 # Neutral court in tourney, slight in regular. Keeping 0 for simplicity.

    # We need to store the Elo of the teams *before* the game occurs
    game_elos = []

    current_season = -1
    season_elo = {}

    for idx, row in df.iterrows():
        season = row['Season']

        # Reset if new season
        if season != current_season:
            current_season = season
            season_elo = {} # Reset ratings for new season

        w_team = row['WTeamID']
        l_team = row['LTeamID']

        # Get current ratings (default 1500)
        elo_w = season_elo.get(w_team, 1500)
        elo_l = season_elo.get(l_team, 1500)

        # Store pre-game elo for feature engineering later
        game_elos.append({
            'Season': season,
            'DayNum': row['DayNum'],
            'WTeamID': w_team,
            'LTeamID': l_team,
            'EloW': elo_w,
            'EloL': elo_l
        })

        # Calculate Expected Score
        expected_w = 1 / (1 + 10 ** ((elo_l - elo_w) / 400))
        expected_l = 1 / (1 + 10 ** ((elo_w - elo_l) / 400))

        # Actual Score (Win = 1, Loss = 0)
        actual_w = 1
        actual_l = 0

        # Update Ratings
        new_elo_w = elo_w + K_FACTOR * (actual_w - expected_w)
        new_elo_l = elo_l + K_FACTOR * (actual_l - expected_l)

        season_elo[w_team] = new_elo_w
        season_elo[l_team] = new_elo_l

    elo_df = pd.DataFrame(game_elos)
    return elo_df

# Calculate Elo for Regular Season (used to build strength before Tourney)
regular_elo = calculate_elo_ratings(regular_season)
print(f"Elo Calculations Generated: {len(regular_elo)} rows")
print(regular_elo.head())

# CREATE FEATURES FOR MATCHUP

In [ ]:
print("\n## Step 4: Create Features")

def prepare_features(tourney_df, team_stats_df, elo_df, seeds_df):
    """
    Creates the final ML dataset with features for Team A and Team B.
    Ensures Team A is the reference (Target = 1 if A wins).
    """
    data = []

    # Merge Seeds
    # Seeds format: "1", "16", etc. Need to extract numeric seed.
    seeds_df['Seed'] = seeds_df['Seed'].str.extract('(\d+)').astype(int)

    # We need to map Elo to the specific time of the tourney game.
    # Since Tourney games happen at end of season, we use the final Regular Season Elo.
    # Group by Season + TeamID to get final Elo
    final_elo = elo_df.sort_values('DayNum').groupby(['Season', 'WTeamID']).last().reset_index()
    final_elo = final_elo.rename(columns={'WTeamID': 'TeamID', 'EloW': 'EloRating'})
    # Also include losers from elo_df to ensure all teams covered
    final_elo_l = elo_df.sort_values('DayNum').groupby(['Season', 'LTeamID']).last().reset_index()
    final_elo_l = final_elo_l.rename(columns={'LTeamID': 'TeamID', 'EloL': 'EloRating'})
    final_elo = pd.concat([final_elo[['Season', 'TeamID', 'EloRating']],
                           final_elo_l[['Season', 'TeamID', 'EloRating']]]).drop_duplicates(subset=['Season', 'TeamID'])

    for idx, row in tourney_df.iterrows():
        season = row['Season']
        team_a = row['WTeamID'] # Winner is Team A for training data
        team_b = row['LTeamID'] # Loser is Team B
        target = 1 # A beat B

        # If we were preparing test set, we wouldn't know W/L, but for training we do.
        # To make model robust, we should ideally randomize A/B or always sort IDs.
        # Standard practice: Always sort TeamIDs so ID1 < ID2, and target indicates if ID1 won.
        # However, for this pipeline, we will keep WTeam as Team A for training clarity,
        # but for Test Set we must follow SampleSubmission ID order.

        # Get Stats
        stats_a = team_stats_df[(team_stats_df['Season']==season) & (team_stats_df['TeamID']==team_a)]
        stats_b = team_stats_df[(team_stats_df['Season']==season) & (team_stats_df['TeamID']==team_b)]

        # Get Elo
        elo_a = final_elo[(final_elo['Season']==season) & (final_elo['TeamID']==team_a)]['EloRating'].values
        elo_b = final_elo[(final_elo['Season']==season) & (final_elo['TeamID']==team_b)]['EloRating'].values

        elo_a = elo_a[0] if len(elo_a) > 0 else 1500
        elo_b = elo_b[0] if len(elo_b) > 0 else 1500

        # Get Seeds
        seed_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team_a)]['Seed'].values
        seed_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team_b)]['Seed'].values

        seed_a = seed_a[0] if len(seed_a) > 0 else 16 # Default to worst seed if missing
        seed_b = seed_b[0] if len(seed_b) > 0 else 16

        # Get Stats values (handle missing)
        win_rate_a = stats_a['WinRate'].values[0] if len(stats_a) > 0 else 0.5
        win_rate_b = stats_b['WinRate'].values[0] if len(stats_b) > 0 else 0.5

        margin_a = stats_a['AvgScoreMargin'].values[0] if len(stats_a) > 0 else 0
        margin_b = stats_b['AvgScoreMargin'].values[0] if len(stats_b) > 0 else 0

        # For Offensive/Defensive Rating, we approximate using PointsScored/Allowed per game
        # Re-calculate from stats df for precision
        gp_a = stats_a['GamesPlayed'].values[0] if len(stats_a) > 0 else 1
        off_a = (stats_a['PointsScored'].values[0] if len(stats_a) > 0 else 0) / gp_a
        def_a = (stats_a['PointsAllowed'].values[0] if len(stats_a) > 0 else 0) / gp_a

        gp_b = stats_b['GamesPlayed'].values[0] if len(stats_b) > 0 else 1
        off_b = (stats_b['PointsScored'].values[0] if len(stats_b) > 0 else 0) / gp_b
        def_b = (stats_b['PointsAllowed'].values[0] if len(stats_b) > 0 else 0) / gp_b

        # Recent Win Rate (Simplified: Using Season Win Rate as proxy for 'last 10'
        # as compact results don't easily allow rolling window without heavy compute in this snippet)
        # In a production pipeline, you would iterate last 10 games specifically.
        recent_a = win_rate_a
        recent_b = win_rate_b

        row_features = {
            'Season': season,
            'TeamA': team_a,
            'TeamB': team_b,
            'elo_difference': elo_a - elo_b,
            'seed_difference': seed_b - seed_a, # Lower seed is better, so B - A
            'recent_win_rate_difference': recent_a - recent_b,
            'average_score_margin_difference': margin_a - margin_b,
            'offensive_rating_difference': off_a - off_b,
            'defensive_rating_difference': def_b - def_a, # Lower is better, so B - A
            'Target': target
        }
        data.append(row_features)

    return pd.DataFrame(data)

# Prepare Training Data (Historical Tourney Results)
train_df = prepare_features(tourney_results, team_stats, regular_elo, tourney_seeds)
print(f"Training Features Shape: {train_df.shape}")
print(train_df.head())

# CONVERT TO ML TRAINING DATASET (TEST SET PREP)

In [ ]:
print("\n## Step 5: Prepare Test Dataset")

# Parse Sample Submission IDs to create Test Features
# ID Format: YYYY_XXXX_YYYY_XXXX
def parse_submission_ids(sub_df):
    parsed = []
    for idx, row in sub_df.iterrows():
        id_str = row['ID']
        parts = id_str.split('_')
        season = int(parts[0])
        team_a = int(parts[1])
        team_b = int(parts[2])
        parsed.append({'Season': season, 'TeamA': team_a, 'TeamB': team_b})
    return pd.DataFrame(parsed)

test_ids = parse_submission_ids(sample_sub)

# We need to generate features for test_ids similar to train_df but without Target
# Re-use logic from prepare_features but adapted for unknown outcome
def prepare_test_features(test_df, team_stats_df, elo_df, seeds_df):
    data = []
    # Get final elo map
    final_elo = elo_df.sort_values('DayNum').groupby(['Season', 'WTeamID']).last().reset_index()
    final_elo = final_elo.rename(columns={'WTeamID': 'TeamID', 'EloW': 'EloRating'})
    final_elo_l = elo_df.sort_values('DayNum').groupby(['Season', 'LTeamID']).last().reset_index()
    final_elo_l = final_elo_l.rename(columns={'LTeamID': 'TeamID', 'EloL': 'EloRating'})
    final_elo = pd.concat([final_elo[['Season', 'TeamID', 'EloRating']],
                           final_elo_l[['Season', 'TeamID', 'EloRating']]]).drop_duplicates(subset=['Season', 'TeamID'])

    seeds_df['Seed'] = seeds_df['Seed'].str.extract('(\d+)').astype(int)

    for idx, row in test_df.iterrows():
        season = row['Season']
        team_a = row['TeamA']
        team_b = row['TeamB']

        # Stats
        stats_a = team_stats_df[(team_stats_df['Season']==season) & (team_stats_df['TeamID']==team_a)]
        stats_b = team_stats_df[(team_stats_df['Season']==season) & (team_stats_df['TeamID']==team_b)]

        # Elo
        elo_a = final_elo[(final_elo['Season']==season) & (final_elo['TeamID']==team_a)]['EloRating'].values
        elo_b = final_elo[(final_elo['Season']==season) & (final_elo['TeamID']==team_b)]['EloRating'].values
        elo_a = elo_a[0] if len(elo_a) > 0 else 1500
        elo_b = elo_b[0] if len(elo_b) > 0 else 1500

        # Seeds
        seed_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team_a)]['Seed'].values
        seed_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==team_b)]['Seed'].values
        seed_a = seed_a[0] if len(seed_a) > 0 else 16
        seed_b = seed_b[0] if len(seed_b) > 0 else 16

        # Metrics
        gp_a = stats_a['GamesPlayed'].values[0] if len(stats_a) > 0 else 1
        off_a = (stats_a['PointsScored'].values[0] if len(stats_a) > 0 else 0) / gp_a
        def_a = (stats_a['PointsAllowed'].values[0] if len(stats_a) > 0 else 0) / gp_a
        margin_a = stats_a['AvgScoreMargin'].values[0] if len(stats_a) > 0 else 0
        win_rate_a = stats_a['WinRate'].values[0] if len(stats_a) > 0 else 0.5

        gp_b = stats_b['GamesPlayed'].values[0] if len(stats_b) > 0 else 1
        off_b = (stats_b['PointsScored'].values[0] if len(stats_b) > 0 else 0) / gp_b
        def_b = (stats_b['PointsAllowed'].values[0] if len(stats_b) > 0 else 0) / gp_b
        margin_b = stats_b['AvgScoreMargin'].values[0] if len(stats_b) > 0 else 0
        win_rate_b = stats_b['WinRate'].values[0] if len(stats_b) > 0 else 0.5

        row_features = {
            'ID': f"{season}_{team_a}_{team_b}",
            'elo_difference': elo_a - elo_b,
            'seed_difference': seed_b - seed_a,
            'recent_win_rate_difference': win_rate_a - win_rate_b,
            'average_score_margin_difference': margin_a - margin_b,
            'offensive_rating_difference': off_a - off_b,
            'defensive_rating_difference': def_b - def_a
        }
        data.append(row_features)
    return pd.DataFrame(data)

test_df = prepare_test_features(test_ids, team_stats, regular_elo, tourney_seeds)
print(f"Test Features Shape: {test_df.shape}")

# Define Feature Columns
feature_cols = [
    'elo_difference',
    'seed_difference',
    'recent_win_rate_difference',
    'average_score_margin_difference',
    'offensive_rating_difference',
    'defensive_rating_difference'
]

X_train = train_df[feature_cols]
y_train = train_df['Target']
X_test = test_df[feature_cols]

# 5-FOLD CROSS VALIDATION SETUP

In [ ]:
print("\n## Step 6: Cross Validation Setup")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

test_preds_lgb = np.zeros(len(X_test))
test_preds_xgb = np.zeros(len(X_test))
test_preds_cat = np.zeros(len(X_test))

log_losses = {'LightGBM': [], 'XGBoost': [], 'CatBoost': []}

# TRAIN MODELS & GENERATE OOF PREDICTIONS

In [ ]:
print("\n## Step 7 & 8: Training Models & OOF Predictions")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold + 1} ---")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # 1. LightGBM
    lgb_model = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        verbose=-1
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50)
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS
    log_losses['LightGBM'].append(log_loss(y_val, oof_lgb[val_idx]))

    # 2. XGBoost
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_test)[:, 1] / N_SPLITS
    log_losses['XGBoost'].append(log_loss(y_val, oof_xgb[val_idx]))

    # 3. CatBoost
    cat_model = cb.CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        random_state=RANDOM_STATE,
        verbose=0,
        loss_function='Logloss'
    )
    cat_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=50)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_preds_cat += cat_model.predict_proba(X_test)[:, 1] / N_SPLITS
    log_losses['CatBoost'].append(log_loss(y_val, oof_cat[val_idx]))

# Print CV Scores
print("\n## Cross Validation Log Loss Scores")
for model_name, scores in log_losses.items():
    print(f"{model_name}: {np.mean(scores):.5f} (+/- {np.std(scores):.5f})")


## BUILD ENSEMBLE


In [ ]:
print("\n## Step 9: Build Ensemble")

# Simple Average Ensemble
ensemble_oof = (oof_lgb + oof_xgb + oof_cat) / 3
ensemble_test = (test_preds_lgb + test_preds_xgb + test_preds_cat) / 3

ensemble_score = log_loss(y_train, ensemble_oof)
print(f"Ensemble OOF Log Loss: {ensemble_score:.5f}")

# Check correlation between models (Optional but good practice)
corr_df = pd.DataFrame({'LGB': oof_lgb, 'XGB': oof_xgb, 'CAT': oof_cat})
print("\nModel Prediction Correlations:")
print(corr_df.corr())

# CREATE SUBMISSION FILE

In [ ]:
print("\n## Step 10: Create Submission File")

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Pred': ensemble_test
})

# Ensure prediction is clipped between 0 and 1 (safety measure)
submission['Pred'] = submission['Pred'].clip(0, 1)

# Save
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")
print(submission.head())